# Synthetic Data Generation for RAG Evaluation

Session 1 built a vector RAG application over a cat health guideline PDF. This
session creates an evaluation dataset for that application and uses the dataset
to compare two retrieval configurations. All generation, embedding, RAG, and
judge requests are routed through Vercel AI Gateway.

The workflow is:

~~~text
corpus -> knowledge graph -> synthetic examples -> human review
       -> LangSmith dataset -> baseline and candidate experiments
~~~

Synthetic examples reduce the cost of getting started, but generated references
are not automatically ground truth. We will inspect and curate them before using
them as evaluation targets.

> This is an educational cat health exercise, not veterinary advice. Generated
> questions and answers must not be used to diagnose, prescribe, or replace a
> veterinarian.

## Learning Outcomes

By the end of this notebook, you will be able to:

- Explain how Ragas builds a knowledge graph for test data generation.
- Distinguish single-hop specific, multi-hop specific, and multi-hop abstract queries.
- Generate and review synthetic questions, reference contexts, and reference answers.
- Route generation, embeddings, RAG, and judge calls through Vercel AI Gateway.
- Upload reviewed examples to a LangSmith dataset.
- Evaluate answer correctness, answer groundedness, and retrieval relevance.
- Run a controlled RAG experiment that changes one variable at a time.

## Table of Contents

- **Breakout Room #1: Synthetic Test Data with Ragas**
  - Task 1: Environment Setup
  - Task 2: Load the Cat Health Corpus
  - Task 3: Build and Enrich a Knowledge Graph
  - Task 4: Inspect the Query Distribution
  - Task 5: Generate and Inspect a Synthetic Test Set
  - Activity #1: Review and Curate the Dataset
- **Breakout Room #2: RAG Evaluation with LangSmith**
  - Task 6: Create a LangSmith Dataset
  - Task 7: Build a Baseline RAG Application
  - Task 8: Define RAG Evaluators
  - Task 9: Run the Baseline Experiment
  - Task 10: Change One Retrieval Variable and Re-Evaluate
  - Activity #2: Compare, Diagnose, and Iterate
  - Advanced Build: Add Robustness and Adversarial Cases

---
# Breakout Room #1
## Synthetic Test Data with Ragas

Ragas uses the source corpus to create a richer representation of its topics and
relationships. Query synthesizers then select scenarios from that representation
and generate questions plus reference answers.

The knowledge graph is a generation aid. It is not the graph used by the RAG
application in Breakout Room #2.

## Task 1: Environment Setup

From the <code>05_Synthetic_Data_Generation_for_RAG_Evals</code> folder:

~~~bash
uv sync
~~~

Then select the environment created by uv as this notebook's kernel.

Required accounts:

- Vercel AI Gateway for generation, embeddings, the RAG answer model, and judges
- LangSmith for the dataset and experiments

A direct OpenAI API key is not required. The OpenAI SDK is used only as a
protocol-compatible client pointed at Vercel AI Gateway.

The default synthetic test set is intentionally small. Ragas generation and
LLM-as-judge evaluation both make multiple model calls, so start small and scale
only after inspecting quality.

### Imports

In [1]:
from __future__ import annotations

import os
from collections import Counter
from getpass import getpass
from importlib.metadata import version
from pathlib import Path
from uuid import uuid4

import instructor
from IPython.display import display
from openai import AsyncOpenAI
from pydantic import BaseModel, field_validator

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langsmith import Client, evaluate
from openevals.llm import create_llm_as_judge
from openevals.prompts import (
    CORRECTNESS_PROMPT,
    RAG_GROUNDEDNESS_PROMPT,
    RAG_RETRIEVAL_RELEVANCE_PROMPT,
)

from ragas.embeddings import embedding_factory
from ragas.llms import llm_factory
from ragas.run_config import RunConfig
from ragas.testset import TestsetGenerator
from ragas.testset.graph import KnowledgeGraph, Node, NodeType
from ragas.testset.synthesizers import (
    MultiHopAbstractQuerySynthesizer,
    MultiHopSpecificQuerySynthesizer,
    SingleHopSpecificQuerySynthesizer,
    default_query_distribution,
)
from ragas.testset.transforms import (
    CustomNodeFilter,
    SummaryExtractor,
    apply_transforms,
    default_transforms_for_prechunked,
)

/home/domenico/Projects/The-AI-Engineering-Certification-v1.0/05_Synthetic_Data_Generation_for_RAG_Evals/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### API Keys, Models, and Cost Controls

The notebook reads model names and budgets from environment variables so you can
tune cost without editing every cell. Vercel AI Gateway exposes an
OpenAI-compatible endpoint, so the OpenAI and LangChain clients only need a
different API key, base URL, and provider-qualified model ID.

See the [Vercel AI Gateway Python documentation](https://vercel.com/docs/ai-gateway/sdks-and-apis/python)
for the current authentication and endpoint details.

LangSmith uses <code>LANGSMITH_TRACING</code>. The older
<code>LANGCHAIN_TRACING_V2</code> name from the source notebook is no longer
needed here.

In [3]:
gateway_api_key = (
    os.environ.get("AI_GATEWAY_API_KEY")
    or os.environ.get("VERCEL_OIDC_TOKEN")
)

if not gateway_api_key:
    gateway_api_key = getpass("Vercel AI Gateway API Key: ")
    os.environ["AI_GATEWAY_API_KEY"] = gateway_api_key

if not os.environ.get("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_API_KEY"] = getpass("LangSmith API Key: ")

os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault(
    "LANGSMITH_PROJECT",
    "aim-session-5-synthetic-rag-evals",
)

GATEWAY_BASE_URL = os.environ.get(
    "AI_GATEWAY_BASE_URL",
    "https://ai-gateway.vercel.sh/v1",
)
GENERATOR_MODEL_NAME = os.environ.get(
    "AIM_GENERATOR_MODEL",
    "openai/gpt-5.4-mini",
)
RAG_MODEL_NAME = os.environ.get(
    "AIM_RAG_MODEL",
    "openai/gpt-5.4-mini",
)
JUDGE_MODEL_NAME = os.environ.get(
    "AIM_JUDGE_MODEL",
    "openai/gpt-5.4-mini",
)
EMBEDDING_MODEL_NAME = os.environ.get(
    "AIM_EMBEDDING_MODEL",
    "openai/text-embedding-3-small",
)
TESTSET_SIZE = int(os.environ.get("AIM_TESTSET_SIZE", "6"))
MAX_CONCURRENCY = int(os.environ.get("AIM_MAX_CONCURRENCY", "8"))

gateway_models = {
    "generator": GENERATOR_MODEL_NAME,
    "rag": RAG_MODEL_NAME,
    "judge": JUDGE_MODEL_NAME,
    "embedding": EMBEDDING_MODEL_NAME,
}
for role, model_name in gateway_models.items():
    if "/" not in model_name:
        raise ValueError(
            f"{role} model must use a provider-qualified AI Gateway ID: "
            f"{model_name!r}"
        )

print(f"Ragas: {version('ragas')}")
print(f"LangSmith: {version('langsmith')}")
print(f"AI Gateway: {GATEWAY_BASE_URL}")
print(f"Generator model: {GENERATOR_MODEL_NAME}")
print(f"RAG model: {RAG_MODEL_NAME}")
print(f"Judge model: {JUDGE_MODEL_NAME}")
print(f"Embedding model: {EMBEDDING_MODEL_NAME}")
print(f"Synthetic examples: {TESTSET_SIZE}")
print(f"LangSmith tracing: {os.environ['LANGSMITH_TRACING']}")

Ragas: 0.4.4.dev8+g298b68274
LangSmith: 0.8.18
AI Gateway: https://ai-gateway.vercel.sh/v1
Generator model: openai/gpt-5.4-mini
RAG model: openai/gpt-5.4-mini
Judge model: openai/gpt-5.4-mini
Embedding model: openai/text-embedding-3-small
Synthetic examples: 6
LangSmith tracing: true


## Task 2: Load the Cat Health Corpus

The corpus is the bundled 2021 AAHA/AAFP Feline Life Stage Guidelines PDF used
in Session 1. <code>PyPDFLoader</code> extracts one LangChain document per page,
including page metadata that survives later chunking.

This gives the generator multiple related units to connect:

- hydration and urinary signs
- preventive care and senior care
- dental pain and behavior changes
- monitoring and emergency escalation

In [4]:
corpus_path = Path("data/cat_health_guidelines.pdf")

if not corpus_path.exists():
    raise FileNotFoundError(
        f"Expected the course corpus at {corpus_path.resolve()}"
    )

pdf_loader = PyPDFLoader(str(corpus_path))
source_documents = pdf_loader.load()
source_documents = [
    document
    for document in source_documents
    if len(document.page_content.strip()) >= 200
]

for index, document in enumerate(source_documents):
    page_number = int(document.metadata.get("page", index)) + 1
    document.metadata.update(
        {
            "source": corpus_path.name,
            "document_type": "feline_life_stage_guidelines",
            "page_number": page_number,
        }
    )

print(f"Loaded {len(source_documents)} text-containing PDF pages")
for document in source_documents[:5]:
    page_number = document.metadata["page_number"]
    print(
        f"- page {page_number}: "
        f"{len(document.page_content)} characters"
    )

Loaded 20 text-containing PDF pages
- page 1: 4913 characters
- page 2: 2084 characters
- page 3: 5955 characters
- page 6: 5673 characters
- page 7: 3588 characters


Inspect one PDF page and its metadata. The metadata is useful for debugging,
trace inspection, and explaining where a retrieved chunk came from.

In [5]:
sample_document = source_documents[0]

print(sample_document.metadata)
print()
print(sample_document.page_content[:800])

{'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'source': 'cat_health_guidelines.pdf', 'total_pages': 22, 'page': 0, 'page_label': '1', 'document_type': 'feline_life_stage_guidelines', 'page_number': 1}

VETERINARY PRACTICE GUIDELINES
2021 AAHA/AAFP Feline Life Stage Guidelines*
Jessica Quimby, DVM, PhD, DACVIM y, Shannon Gowland, DVM, DABVP y, Hazel C. Carney, DVM, MS, DABVP,
Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM, Paula Plummer, LVT, VTS (ECC, SAIM), Jodi Westropp,
DVM, PhD, DACVIM
ABSTRACT
The guidelines, authored by a Task Force ofexperts in feline clinical medicine, are an update and extension of the AAFP–AAHA
Feline Life Stage Guidelines published in 2010. The guidelines are published simultaneously in theJournal of Feline Medicine and
Surgery(volume 23, issue 3, pages 211–233, DOI: 10.1177

## Task 3: Build and Enrich a Knowledge Graph

The unrolled workflow makes the generation stages visible:

1. Treat each text-containing PDF page as a pre-chunked Ragas node.
2. Run Ragas extractors, embeddings, and relationship builders.
3. Save the graph so expensive enrichment can be reused.

Ragas remains responsible for graph enrichment and synthetic generation. The
newer pinned Ragas build exposes an official Instructor mode parameter, so its
LLM factory can use AI Gateway tool calls directly without custom wrappers.

In [6]:
gateway_client = AsyncOpenAI(
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
)

generator_llm = llm_factory(
    GENERATOR_MODEL_NAME,
    provider="openai",
    client=gateway_client,
    mode=instructor.Mode.TOOLS,
    max_tokens=4096,
)
# Provider-qualified Gateway IDs bypass Ragas's GPT-5 parameter detection.
# Keep only the token limit supported by the Gateway route. max_retries is
# consumed locally by Instructor and is not sent to AI Gateway.
generator_llm.model_args = {
    "max_tokens": 4096,
    "max_retries": 3,
}

generator_embeddings = embedding_factory(
    "openai",
    model=EMBEDDING_MODEL_NAME,
    client=gateway_client,
)

ragas_run_config = RunConfig(
    timeout=180,
    max_retries=3,
    max_wait=30,
    max_workers=MAX_CONCURRENCY,
)

/tmp/ipykernel_2523998/1946522596.py:21: DeprecationWarning: Importing embedding_factory from ragas.embeddings is deprecated. Import directly from ragas.embeddings.base or use modern providers: from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = embedding_factory(


Before building the graph, make one small structured-output request through
Ragas. This catches gateway authentication, model availability, and tool-calling
incompatibilities without waiting for every PDF page to retry.

In [7]:
class GatewayToolCallCheck(BaseModel):
    status: str


class NonEmptySummary(BaseModel):
    text: str

    @field_validator("text")
    @classmethod
    def require_text(cls, value):
        value = value.strip()
        if not value:
            raise ValueError("summary text cannot be empty")
        return value


gateway_check = generator_llm.generate(
    "Use the required tool with a short, non-empty status message.",
    GatewayToolCallCheck,
)
if not gateway_check.status.strip():
    raise RuntimeError("AI Gateway returned an empty tool-call check")

print(f"AI Gateway tool-based structured output: {gateway_check.status}")

AI Gateway tool-based structured output: ok


In [8]:
def build_prechunked_knowledge_graph(chunks):
    return KnowledgeGraph(
        nodes=[
            Node(
                type=NodeType.CHUNK,
                properties={
                    "page_content": chunk.page_content,
                    "document_metadata": dict(chunk.metadata),
                },
            )
            for chunk in chunks
            if chunk.page_content.strip()
        ]
    )


generation_chunks = list(source_documents)
knowledge_graph = build_prechunked_knowledge_graph(generation_chunks)

print(f"Ragas input chunks: {len(generation_chunks)}")
print(knowledge_graph)

Ragas input chunks: 20
KnowledgeGraph(nodes: 20, relationships: 0)


### Apply Ragas Transforms

Because the PDF loader already gives us coherent page-level chunks, use Ragas'
built-in pre-chunked transform pipeline. It skips headline extraction and
splitting, then applies Ragas summaries, embeddings, themes, named entities,
and relationship builders directly to the PDF pages. The parent-child node
filter is omitted because these page chunks intentionally have no parent nodes.
A non-empty output constraint makes Instructor retry blank Ragas summaries before
the built-in embedding transform runs.

In [8]:
knowledge_graph = build_prechunked_knowledge_graph(generation_chunks)
transforms = [
    transform
    for transform in default_transforms_for_prechunked(
        llm=generator_llm,
        embedding_model=generator_embeddings,
    )
    if not isinstance(transform, CustomNodeFilter)
]

summary_transform = next(
    transform
    for transform in transforms
    if isinstance(transform, SummaryExtractor)
)
summary_transform.prompt.output_model = NonEmptySummary

print("Ragas transform pipeline:")
for transform in transforms:
    nested = getattr(transform, "transformations", None)
    if nested is None:
        print(f"- {type(transform).__name__}")
    else:
        names = ", ".join(type(item).__name__ for item in nested)
        print(f"- Parallel({names})")

for transform in transforms:
    apply_transforms(
        knowledge_graph,
        transform,
        run_config=ragas_run_config,
    )
    if isinstance(transform, SummaryExtractor):
        empty_summary_nodes = [
            node
            for node in knowledge_graph.nodes
            if not str(node.get_property("summary") or "").strip()
        ]
        if empty_summary_nodes:
            raise RuntimeError(
                "Ragas did not produce non-empty summaries for "
                f"{len(empty_summary_nodes)} PDF chunks"
            )

print(knowledge_graph)

Ragas transform pipeline:
- SummaryExtractor
- Parallel(EmbeddingExtractor, ThemesExtractor, NERExtractor)
- Parallel(CosineSimilarityBuilder, OverlapScoreBuilder)


Applying SummaryExtractor:   0%|          | 0/20 [00:00<?, ?it/s]

Applying SummaryExtractor: 100%|██████████| 20/20 [06:35<00:00, 19.76s/it]
Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:  25%|██▌       | 15/60 [02:53<10:23, 13.85s/it]Task failed with APITimeoutError: Request timed out.
Task failed with APITimeoutError: Request timed out.
Task failed with APITimeoutError: Request timed out.
Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]: 100%|██████████| 60/60 [15:58<00:00, 15.98s/it]


APITimeoutError: Request timed out.

Inspect the graph at a high level. Different Ragas versions may add different
properties, so the notebook avoids depending on one exact internal schema.

In [ ]:
node_type_counts = Counter(str(node.type) for node in knowledge_graph.nodes)

print("Node types:")
for node_type, count in node_type_counts.items():
    print(f"- {node_type}: {count}")

print(f"Relationships: {len(knowledge_graph.relationships)}")

for index, node in enumerate(knowledge_graph.nodes[:3], start=1):
    property_names = sorted(node.properties.keys())
    print(f"Node {index} properties: {property_names}")

Node types:
- NodeType.CHUNK: 20
Relationships: 0
Node 1 properties: ['document_metadata', 'entities', 'page_content', 'summary', 'summary_embedding', 'themes']
Node 2 properties: ['document_metadata', 'entities', 'page_content', 'summary', 'summary_embedding', 'themes']
Node 3 properties: ['document_metadata', 'entities', 'page_content', 'summary', 'summary_embedding', 'themes']


### Save and Reload the Graph

Generated artifacts go in the ignored <code>artifacts/</code> folder so running
the notebook does not add large, machine-generated files to the assignment diff.

In [15]:
artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(exist_ok=True)

knowledge_graph_path = artifacts_dir / "cat_health_knowledge_graph.json"
#knowledge_graph.save(str(knowledge_graph_path))

loaded_knowledge_graph = KnowledgeGraph.load(str(knowledge_graph_path))

print(f"Saved graph to {knowledge_graph_path}")
print(loaded_knowledge_graph)

Saved graph to artifacts/cat_health_knowledge_graph.json
KnowledgeGraph(nodes: 20, relationships: 41)


#### ❓ Question #1

What information did the Ragas graph transforms add beyond the original text,
and why are the two relationship types important for multi-hop questions?

##### ✅ Answer

It added summaries for each chunk and their embeddings, themes  and extracted entities for each nodes. Examples of entities are the names of the authors or the journal. Examples of themes are Nutrition and weight management and Oral health. 
It also added two relationships (summary similarity and entitie4s overlap) that are relvant for multi-hop qusstions because it allows RAGAS to elaborate questions thata require multiple chunks to be answered in a complete way thus testing our RAG pipeline on realistic scenarios. Having two kinds of relationships is important because they capture different kind of connections.

## Task 4: Inspect the Query Distribution

Ragas can synthesize several kinds of questions:

| Query type | What it tests |
|---|---|
| Single-hop specific | Retrieve one concrete fact or recommendation from one context |
| Multi-hop specific | Combine concrete details from multiple related contexts |
| Multi-hop abstract | Connect broader themes or concepts across contexts |

The distribution is part of the evaluation specification. It determines which
behaviors are common in the generated dataset.

In [12]:
query_distribution = default_query_distribution(
    generator_llm,
    kg=loaded_knowledge_graph,
)

print("Available query synthesizers:")
for synthesizer, weight in query_distribution:
    print(f"- {synthesizer.name}: {weight:.0%}")

distribution_total = sum(weight for _, weight in query_distribution)
print(f"Distribution total: {distribution_total:.2f}")

Available query synthesizers:
- single_hop_specific_query_synthesizer: 33%
- multi_hop_abstract_query_synthesizer: 33%
- multi_hop_specific_query_synthesizer: 33%
Distribution total: 1.00


### Define a Custom Distribution

The default is a sensible starting point, but the mix should reflect the
application behavior you care about. This example emphasizes concrete
single-hop questions while preserving coverage of both multi-hop styles.

Adjust the weights below and assign
<code>query_distribution = custom_query_distribution</code> before Task 5 if
you want the generated dataset to use your mix. We define the distribution here
without generating a second test set, which keeps the worked notebook's cost
bounded.

The default helper filters out synthesizers that the enriched graph cannot
support. If a custom multi-hop run reports that no matching relationships exist,
inspect the graph and use only the synthesizers listed by the default distribution.

In [13]:
custom_query_distribution = [
    (
        SingleHopSpecificQuerySynthesizer(llm=generator_llm),
        0.50,
    ),
    (
        MultiHopSpecificQuerySynthesizer(llm=generator_llm),
        0.30,
    ),
    (
        MultiHopAbstractQuerySynthesizer(llm=generator_llm),
        0.20,
    ),
]

assert abs(
    sum(weight for _, weight in custom_query_distribution) - 1.0
) < 1e-9

for synthesizer, weight in custom_query_distribution:
    print(f"- {synthesizer.name}: {weight:.0%}")

- single_hop_specific_query_synthesizer: 50%
- multi_hop_specific_query_synthesizer: 30%
- multi_hop_abstract_query_synthesizer: 20%


#### ❓ Question #2

Describe the three query types in your own words. Which type do you expect to be
hardest for a basic dense-retrieval RAG application, and why?

##### ✅ Answer

- Single hop specific is a questionabout a specific fact that can be answered having in context the content of a singne node of the knowledge graph (single chunk in our case)
  - For example a question of the kind "when were AAFP–AAHA Feline Life Stage Guidelines proposed ? " the answer will most likely be in a single chunk in a sentence with the year (example from above sample text)
- Multi hop specific questions need specific context from multiple nodes in the knowledge graph.
  -  An example could be "what are the differences between AAFP–AAHA Feline Life Stage Guidelines  and alternative guidelines X ?"
- Multi hop abstract need multiple nodes but have a less specific connotation and require association between the abstract themes being asked about and the specific contexts available in the chunks
  -  For example "how many official guidelines of relevance for cats' healt were released in the last 20 years and what are their differences ?"


Multi hop abstract is the hardest for a basic dense retrieval RAG application because they tend to retrieve chunks that are directly semantically close to the query. 


## Task 5: Generate and Inspect a Synthetic Test Set

Each generated row contains:

- <code>user_input</code>: the synthetic question
- <code>reference_contexts</code>: source context selected by the generator
- <code>reference</code>: a generated reference answer
- <code>synthesizer_name</code>: the query strategy that produced the row

The reference is generated from selected source context. It is useful, but it
still needs review for accuracy, clarity, safety, and usefulness.

In [12]:
testset_generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
    knowledge_graph=loaded_knowledge_graph,
)

synthetic_testset = testset_generator.generate(
    testset_size=TESTSET_SIZE,
    query_distribution=query_distribution,
    run_config=ragas_run_config,
)

testset_df = synthetic_testset.to_pandas()

display(
    testset_df[
        [
            "user_input",
            "reference",
            "synthesizer_name",
        ]
    ]
)

Generating Samples: 100%|██████████| 6/6 [02:01<00:00, 20.22s/it]


,user_input,reference,synthesizer_name
0,What is Jessica Quimby’s role in the 2021 AAHA...,"Jessica Quimby, DVM, PhD, DACVIM, is listed am...",single_hop_specific_query_synthesizer
1,What does AAHA provide for cats' life stages?,The 2021 AAHA/AAFP Feline Life Stage Guideline...,single_hop_specific_query_synthesizer
2,"How do ""appetite and elimination changes"" guid...","For mature adult and senior cats, specific que...",multi_hop_abstract_query_synthesizer
3,In the context of diabetes management and body...,The context identifies 2018 AAHA diabetes mana...,multi_hop_abstract_query_synthesizer
4,According to the FIC and urine marking themes ...,The context says that although urine-marking b...,multi_hop_specific_query_synthesizer
5,What do the feline life stage guidelines say t...,"For Young Adult Cats, the guidelines recommend...",multi_hop_specific_query_synthesizer


In [ ]:
testset_path = artifacts_dir / "cat_health_synthetic_testset.jsonl"
testset_df.to_json(
    testset_path,
    orient="records",
    lines=True,
    force_ascii=False,
)

print("Examples by synthesizer:")
print(testset_df["synthesizer_name"].value_counts())
print()
print(f"Saved candidate examples to {testset_path}")

Examples by synthesizer:

Saved candidate examples to artifacts/cat_health_synthetic_testset.jsonl


### Abstracted Ragas Alternative

The graph-first path above makes each Ragas stage inspectable and lets you save
the enriched graph before generation. Ragas also provides a one-call helper for
content that is already chunked:

~~~python
quick_generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
)
quick_testset = quick_generator.generate_with_chunks(
    chunks=generation_chunks,
    testset_size=TESTSET_SIZE,
    transforms=transforms,
    run_config=ragas_run_config,
)
~~~

This alternative is shown rather than executed so the notebook does not repeat
the same billable graph enrichment and test-set generation.

#### ❓ Question #3

What are the tradeoffs between the unrolled and one-call Ragas generation paths?
When would you choose each one?

##### ✅ Answer

The **unrolled** path exposes each stage — transforms, relationship building, embeddings — so you can tune parameters individually, inspect intermediate graph state, cache expensive steps, and repair or partially re-run a failed stage. The cost is more code and more moving parts to manage.

The **one-call** path (`generate_with_langchain_docs`) wraps all of that behind a single call: less code, faster to stand up, and a sensible default pipeline — at the cost of visibility and fine-grained control.

**When to choose each:** the one-call path for a quick first dataset or when the defaults are good enough; the unrolled path when you need to customize transforms, debug graph quality, control cost, or reproduce/repair runs deterministically.

## 🏗️ Activity #1: Review and Curate the Dataset

Review every generated row before uploading it.

For each example, check:

1. Is the question answerable from the reference contexts?
2. Is the reference answer fully supported by those contexts?
3. Is the wording natural for a plausible user?
4. Does the example duplicate another row?
5. Does it preserve the corpus's medical-safety boundaries?

Requirements:

- Remove or repair at least one weak example, if one exists.
- Record why you kept, edited, or removed it.
- Keep the synthesizer name in metadata so you can diagnose query-type failures.

In [23]:
# Activity #1 workspace
#
# Start with every generated example. Replace this with your reviewed subset.
approved_testset_df = testset_df.copy()
review_status = "review_required"

# Examples:
# approved_testset_df = testset_df.drop(index=[2]).reset_index(drop=True)
# approved_testset_df.loc[0, "user_input"] = "Your clearer question"
# approved_testset_df.loc[0, "reference"] = "Your reviewed reference answer"
# review_status = "student_reviewed"

approved_testset_df = approved_testset_df.drop(index=[0])

approved_testset_df.loc[2, "user_input"] = (
    "Which appetite and elimination changes should guide the examination of "
    "adult cats, and which congenital abnormalities should be checked in kittens?"
)


approved_testset_df = approved_testset_df.reset_index(drop=True)
review_status = "student_reviewed"




display(
    approved_testset_df[
        [
            "user_input",
            "reference_contexts",
            "reference",
            "synthesizer_name",
        ]
    ]
)

,user_input,reference_contexts,reference,synthesizer_name
0,What does AAHA provide for cats' life stages?,[Introduction\nThe feline patient ’s life stag...,The 2021 AAHA/AAFP Feline Life Stage Guideline...,single_hop_specific_query_synthesizer
1,Which appetite and elimination changes should ...,[<1-hop>\n\ndetection of changes and identi ﬁc...,"For mature adult and senior cats, specific que...",multi_hop_abstract_query_synthesizer
2,In the context of diabetes management and body...,"[<1-hop>\n\n130. van Bree FPJ, Bokken GCAM, Mi...",The context identifies 2018 AAHA diabetes mana...,multi_hop_abstract_query_synthesizer
3,According to the FIC and urine marking themes ...,"[<1-hop>\n\n10 months, primarily by phone cont...",The context says that although urine-marking b...,multi_hop_specific_query_synthesizer
4,What do the feline life stage guidelines say t...,[<1-hop>\n\ndetection of changes and identi ﬁc...,"For Young Adult Cats, the guidelines recommend...",multi_hop_specific_query_synthesizer


### 📝 Activity #1 Notes

- Example reviewed: "What is Jessica Quimby's role…?"
- Decision: Remove
- Reason: Feels an irrelevant question. Why would a user be interested in the role of a specific author ?
- Any safety or grounding issue found: No

- Example reviewed:"What does AAHA provide for cats' life stages?" 
- Decision: Keep
- Reason: Feels like a plausible question
- Any safety or grounding issue found: No

- Example reviewed: How do "appetite and elimination changes" guide examination of adult cats, and what "congenital abnormalities" should be checked in kittens?
- Decision: Repair
- Reason: Wording feels unnatural and the multi hop nature a bit forced.
- Any safety or grounding issue found: no 

- Example reviewed: "In the context of diabetes management and body condition scoring and obesity, how do the cited feline guidelines connect diabetes care with weight assessment in cats?"
- Decision: Keep
- Reason: Seems plausible
- Any safety or grounding issue found: No

The remaining examples were all kept for the same reason as the last one.

---
# Breakout Room #2
## RAG Evaluation with LangSmith

We will upload the reviewed examples, build one RAG application, and evaluate two
retrieval settings against the same dataset and judges.

Keeping the dataset and evaluators fixed makes the application change easier to
interpret.

## Task 6: Create a LangSmith Dataset

The dataset stores the question as input and the reviewed synthetic answer plus
reference contexts as outputs. Query type and provenance remain metadata.

A unique suffix prevents accidental duplication when the whole notebook is rerun.
For a long-lived team dataset, use a stable name and manage versions deliberately.

In [34]:
import hashlib

content_hash = hashlib.sha256(
    approved_testset_df[
        ["user_input", "reference", "reference_contexts", "synthesizer_name"]
    ].to_json(orient="records").encode("utf-8")
).hexdigest()[:12]

def as_string_list(value) -> list[str]:
    if value is None:
        return []
    if isinstance(value, list):
        return [str(item) for item in value]
    if hasattr(value, "tolist"):
        converted = value.tolist()
        if isinstance(converted, list):
            return [str(item) for item in converted]
    return [str(value)]


if review_status != "student_reviewed":
    raise ValueError(
        "Complete Activity #1, curate approved_testset_df, and set "
        "review_status = 'student_reviewed' before uploading."
    )



langsmith_client = Client()

dataset_name = f"aim-session-5-cat-health-synthetic-{content_hash}"

dataset_exists = langsmith_client.has_dataset(dataset_name=dataset_name)
if dataset_exists:
    langsmith_dataset = langsmith_client.read_dataset(dataset_name=dataset_name)
else:
    langsmith_dataset = langsmith_client.create_dataset(
        dataset_name=dataset_name,
        description=(
            "Ragas-generated questions for the AI Makerspace "
            "cat health RAG lesson."
        ),
        metadata={
            "session": 5,
            "source": "ragas",
            "corpus": str(corpus_path),
        },
    )

langsmith_examples = []
for _, row in approved_testset_df.iterrows():
    langsmith_examples.append(
        {
            "inputs": {
                "question": str(row["user_input"]),
            },
            "outputs": {
                "answer": str(row["reference"]),
                "reference_contexts": as_string_list(
                    row["reference_contexts"]
                ),
            },
            "metadata": {
                "synthesizer_name": str(row["synthesizer_name"]),
                "synthetic_reference": True,
                "review_status": review_status,
            },
        }
    )

if not dataset_exists:
    langsmith_client.create_examples(
        dataset_id=langsmith_dataset.id,
        examples=langsmith_examples,
    )


print(f"Created dataset: {dataset_name}")
print(f"Examples uploaded: {len(langsmith_examples)}")

Created dataset: aim-session-5-cat-health-synthetic-51c70f65127b
Examples uploaded: 5


#### ❓ Question #4

Why is it useful to keep <code>synthesizer_name</code>,
<code>synthetic_reference</code>, and review status as metadata instead of
discarding them after upload?

##### ✅ Answer

It iseful for future inspection and evolution of the dataset. Knowing if a row was synthetically generated, how it was generated and wether it was reviewed by a human is crucial information to track.  Human curated examples tipically bring more value as they are more likely to be representative of real users' behaviour. 

## Task 7: Build a Baseline RAG Application

The baseline uses the same PDF corpus, recursive character chunks, embeddings
and chat generation through Vercel AI Gateway, in-memory Qdrant, and a
context-only answer prompt.

The target returns both the answer and the retrieved contexts. Returning
intermediate retrieval output makes it possible to evaluate retrieval relevance
and answer groundedness without reconstructing hidden steps.

In [27]:
rag_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=75,
)
rag_documents = rag_splitter.split_documents(source_documents)

rag_embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL_NAME,
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
    check_embedding_ctx_length=False,
)
vector_store = QdrantVectorStore.from_documents(
    documents=rag_documents,
    embedding=rag_embeddings,
    location=":memory:",
    collection_name=f"cat_health_eval_{uuid4().hex[:8]}",
)

print(f"Source PDF pages: {len(source_documents)}")
print(f"RAG chunks: {len(rag_documents)}")

Source PDF pages: 20
RAG chunks: 255


In [28]:
RAG_SYSTEM_PROMPT = """You are an educational cat health assistant.

Answer the question using only the retrieved context.
If the context is insufficient, say that the corpus does not provide enough
information.

Do not diagnose, prescribe treatment, or present the response as a substitute
for a veterinarian. Clearly preserve any urgent-care guidance found in the
context.

Retrieved context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", RAG_SYSTEM_PROMPT),
        ("human", "{question}"),
    ]
)
rag_llm = ChatOpenAI(
    model=RAG_MODEL_NAME,
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
)
answer_chain = rag_prompt | rag_llm | StrOutputParser()

In [35]:
def format_retrieved_document(document) -> str:
    page_number = document.metadata.get("page_number", "unknown")
    source = document.metadata.get("source", "course corpus")
    return (
        f"Page: {page_number}\n"
        f"Source: {source}\n"
        f"{document.page_content}"
    )


def make_rag_target(retrieval_k: int):
    retriever = vector_store.as_retriever(
        search_kwargs={"k": retrieval_k}
    )

    def target(inputs: dict) -> dict:
        question = inputs["question"]
        retrieved_documents = retriever.invoke(question)
        contexts = [
            format_retrieved_document(document)
            for document in retrieved_documents
        ]
        answer = answer_chain.invoke(
            {
                "question": question,
                "context": "\n\n".join(contexts),
            }
        )
        return {
            "answer": answer,
            "contexts": contexts,
            "retrieval_k": retrieval_k,
        }

    target.__name__ = f"cat_health_rag_k_{retrieval_k}"
    return target

In [37]:
baseline_retrieval_k = 3
baseline_target = make_rag_target(baseline_retrieval_k)

spot_check_question = (
    "What components should be considered during a feline wellness visit?"
)
baseline_spot_check = baseline_target(
    {"question": spot_check_question}
)

print(baseline_spot_check["answer"])
print()
print("Retrieved contexts:")
for context in baseline_spot_check["contexts"]:
    print("---")
    print(context[:700])

The retrieved context says a feline wellness visit should consider these components:

- Behavior and environmental needs
- Elimination
- Nutrition and weight management
- Oral health
- Parasite control
- Vaccination
- Zoonoses and human safety
- Diagnostics

It also notes additional important topics such as:

- Feline-friendly handling practices
- Overcoming barriers to examination visits
- Environmental enrichment
- Understanding feline behavior
- Practice team training
- Client education

The corpus does not provide more detail beyond this list.

Retrieved contexts:
---
Page: 1
Source: cat_health_guidelines.pdf
lifelong feline healthcare strategy. The guidelines include a comprehensive table on the components of a feline wellness visit that
provides a framework for systematically implementing an individualized life stage approach to fe line healthcare. Included are
recommendations for managing the most critical health-related factors in relation to a cat’s life stage. These recommend

## Task 8: Define RAG Evaluators

We will evaluate three different relationships:

| Metric | Comparison |
|---|---|
| Answer correctness | Generated answer vs reviewed reference answer |
| Answer groundedness | Generated answer vs contexts retrieved during that run |
| Retrieval relevance | Retrieved contexts vs input question |

These can disagree. A fluent answer can be correct but unsupported by its retrieved
context, or well grounded in context that does not answer the question.

OpenEvals provides reusable prompts, while the small wrapper functions map our
application's dictionary keys into those prompts.

In [38]:
gateway_judge_llm = ChatOpenAI(
    model=JUDGE_MODEL_NAME,
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
)

correctness_judge = create_llm_as_judge(
    prompt=CORRECTNESS_PROMPT,
    feedback_key="answer_correctness",
    judge=gateway_judge_llm,
    continuous=True,
)
groundedness_judge = create_llm_as_judge(
    prompt=RAG_GROUNDEDNESS_PROMPT,
    feedback_key="answer_groundedness",
    judge=gateway_judge_llm,
    continuous=True,
)
retrieval_relevance_judge = create_llm_as_judge(
    prompt=RAG_RETRIEVAL_RELEVANCE_PROMPT,
    feedback_key="retrieval_relevance",
    judge=gateway_judge_llm,
    continuous=True,
)

In [39]:
def answer_correctness(
    inputs: dict,
    outputs: dict,
    reference_outputs: dict,
) -> dict:
    return correctness_judge(
        inputs=inputs["question"],
        outputs=outputs["answer"],
        reference_outputs=reference_outputs["answer"],
    )


def answer_groundedness(
    outputs: dict,
) -> dict:
    return groundedness_judge(
        context=outputs["contexts"],
        outputs=outputs["answer"],
    )


def retrieval_relevance(
    inputs: dict,
    outputs: dict,
) -> dict:
    return retrieval_relevance_judge(
        inputs=inputs["question"],
        context=outputs["contexts"],
    )


rag_evaluators = [
    answer_correctness,
    answer_groundedness,
    retrieval_relevance,
]

#### ❓ Question #5

Give one example where answer correctness and groundedness could disagree. What
would that disagreement tell you to inspect in the trace?

##### ✅ Answer

For example, correctness might be off because the retrieved context that the evaluated response is based upon is off, but groundedness in that case could still be good. If an answer is in agreement with the wrong context, it's not correct, but it is still grounded. So I would inspect the retrieved context and the reasons for it being off. 

## Task 9: Run the Baseline Experiment

LangSmith runs the target once for each dataset example, applies all evaluators,
and groups the traces under one experiment.

After the run, open the experiment URL and inspect individual failures. Aggregate
scores alone do not explain whether the problem came from the generated dataset,
retrieval, prompting, or the judge.

In [40]:
baseline_results = evaluate(
    baseline_target,
    data=dataset_name,
    evaluators=rag_evaluators,
    experiment_prefix="cat-health-rag-baseline-k3",
    description=(
        "Baseline cat health RAG with 500-character chunks "
        "and retrieval k=3."
    ),
    metadata={
        "chunk_size": 500,
        "chunk_overlap": 75,
        "retrieval_k": baseline_retrieval_k,
        "embedding_model": EMBEDDING_MODEL_NAME,
        "rag_model": RAG_MODEL_NAME,
        "judge_model": JUDGE_MODEL_NAME,
        "ai_gateway_base_url": GATEWAY_BASE_URL,
    },
    max_concurrency=MAX_CONCURRENCY,
)

print(f"Baseline experiment: {baseline_results.experiment_name}")

View the evaluation results for experiment: 'cat-health-rag-baseline-k3-5dd091f1' at:
https://smith.langchain.com/o/335c9b89-b5ea-477d-9951-61da5bf5e012/datasets/0bc3e7d9-41f6-4a3a-9619-bd187cf39eb9/compare?selectedSessions=dc871698-0097-4088-a79c-76349dfee290




5it [00:14,  2.83s/it]

Baseline experiment: cat-health-rag-baseline-k3-5dd091f1


### Baseline Inspection Notes

- Lowest-scoring example: What do the feline life stage guidelines say to ask and monitor in Young Adult Cats when evaluating cough, vomiting or vomiting hairballs, diarrhea, weight, and early disease signs, and how do these recommendations relate to the guidance on managing Young Adult Cats with ideal body condition and energy needs?
- Metric that failed: Correctdness
- Was the synthetic reference valid? YES
- Was the retrieved context relevant and sufficient? It was not.
- Did the answer add unsupported information? No

## Task 10: Change One Retrieval Variable and Re-Evaluate

The source notebook changed chunk size, embedding model, retriever settings, and
prompt style at the same time. That makes any score change hard to explain.

Here we change only retrieval depth:

~~~text
baseline:  k = 3
candidate: k = 6
~~~

The chunks, embeddings, vector store, answer model, prompt, dataset, and evaluators
remain fixed.

In [41]:
candidate_retrieval_k = 6
candidate_target = make_rag_target(candidate_retrieval_k)

candidate_spot_check = candidate_target(
    {"question": spot_check_question}
)

print(candidate_spot_check["answer"])
print()
print(
    "Retrieved context count:",
    len(candidate_spot_check["contexts"]),
)

According to the retrieved guidelines, a feline wellness visit should consider these components:

- Current history and life stage
- Physical examination and body condition
- Behavioral and environmental needs
- Elimination
- Nutrition and weight management
- Oral health
- Parasite control
- Vaccination
- Zoonoses and human safety
- Diagnostics

The guidelines also note additional important topics such as feline-friendly handling practices, overcoming barriers to examination visits, environmental enrichment, understanding feline behavior, practice team training, and client education.

If you want, I can also summarize how the visit focus changes by life stage.

Retrieved context count: 6


In [42]:
candidate_results = evaluate(
    candidate_target,
    data=dataset_name,
    evaluators=rag_evaluators,
    experiment_prefix="cat-health-rag-candidate-k6",
    description=(
        "Candidate cat health RAG with the same index and "
        "retrieval k increased from 3 to 6."
    ),
    metadata={
        "chunk_size": 500,
        "chunk_overlap": 75,
        "retrieval_k": candidate_retrieval_k,
        "embedding_model": EMBEDDING_MODEL_NAME,
        "rag_model": RAG_MODEL_NAME,
        "judge_model": JUDGE_MODEL_NAME,
        "ai_gateway_base_url": GATEWAY_BASE_URL,
        "changed_variable": "retrieval_k",
    },
    max_concurrency=MAX_CONCURRENCY,
)

print(f"Candidate experiment: {candidate_results.experiment_name}")

View the evaluation results for experiment: 'cat-health-rag-candidate-k6-71755adc' at:
https://smith.langchain.com/o/335c9b89-b5ea-477d-9951-61da5bf5e012/datasets/0bc3e7d9-41f6-4a3a-9619-bd187cf39eb9/compare?selectedSessions=8763e9fc-5863-46dd-8102-ce1c9afcb206




5it [00:09,  1.88s/it]

Candidate experiment: cat-health-rag-candidate-k6-71755adc


#### ❓ Question #6

Why is changing one variable at a time useful? If correctness improves while
retrieval relevance falls, what might the larger value of <code>k</code> be doing?

##### ✅ Answer

Changing one variable at a time allows us to understand the effect of that dimension of tuning on the quality of the results. If we were to change multiple variables at a single time and observe an improvement, we wouldn't know what we could attribute that hence, we would not know what would be the appropriate next iteration to further improve.

If correctness rises while retrieval relevance falls, the larger `k` is likely trading precision for recall: pulling in more chunks raises the chance the answer-bearing chunk is present (helping correctness), but it also admits more loosely-related chunks that lower the average relevance of the retrieved set. In other words, the model is finding the answer inside a noisier, lower-precision context window.

## 🏗️ Activity #2: Compare, Diagnose, and Iterate

Compare the baseline and candidate experiments in LangSmith.

Requirements:

1. Record the mean score for each evaluator in both experiments.
2. Inspect at least two examples whose scores changed.
3. Decide whether <code>k=6</code> improved the application overall.
4. Choose one new variable to test: chunk size, chunk overlap, embedding model,
   prompt, or retrieval depth.
5. State your prediction before running the experiment.
6. Run a third experiment and explain the result.

Keep the reviewed dataset and evaluators fixed. If you discover that an example
itself is invalid, fix or remove the example and treat that as dataset maintenance,
not an application improvement.

In [43]:
# Activity #2 workspace
#
# A retrieval-depth experiment can reuse the existing vector store:
# student_retrieval_k = 4
# student_target = make_rag_target(student_retrieval_k)
#
# If you change chunking or the embedding model, build a new vector store,
# then create a target with the same output contract:
# {
#     "answer": str,
#     "contexts": list[str],
#     "retrieval_k": int,
# }
#
# Run evaluate(...) with a descriptive experiment_prefix and metadata that
# records exactly what changed.

student_retrieval_k = 9
student_target = make_rag_target(student_retrieval_k)

student_results = evaluate(
    student_target,
    data=dataset_name,
    evaluators=rag_evaluators,
    experiment_prefix=f"cat-health-rag-student-k{student_retrieval_k}",
    description=(
        "Third experiment: same index, retrieval k increased to "
        f"{student_retrieval_k} to test whether more context keeps helping "
        "correctness or starts hurting retrieval relevance."
    ),
    metadata={
        "chunk_size": 500,
        "chunk_overlap": 75,
        "retrieval_k": student_retrieval_k,
        "embedding_model": EMBEDDING_MODEL_NAME,
        "rag_model": RAG_MODEL_NAME,
        "judge_model": JUDGE_MODEL_NAME,
        "ai_gateway_base_url": GATEWAY_BASE_URL,
        "changed_variable": "retrieval_k",
    },
    max_concurrency=MAX_CONCURRENCY,
)

print(f"Student experiment: {student_results.experiment_name}")


View the evaluation results for experiment: 'cat-health-rag-student-k9-e6511641' at:
https://smith.langchain.com/o/335c9b89-b5ea-477d-9951-61da5bf5e012/datasets/0bc3e7d9-41f6-4a3a-9619-bd187cf39eb9/compare?selectedSessions=a480e2b7-6d68-4ff9-a12d-29107163574f




5it [00:09,  1.98s/it]

Student experiment: cat-health-rag-student-k9-e6511641


### 📝 Activity #2 Notes

- Variable changed: retrieval_k
- Prediction: I expect this to improve correctdness as more context being available would improve response quality at such low k values.
- Baseline result (k=3): correctness 0.66, groundedness 0.93, retrieval relevance 0.85.
- Candidate result (k=6): correctness 0.68, groundedness 0.94, retrieval relevance 0.94.
- Third experiment result (k=9): correctness 0.77, groundedness 0.92, retrieval relevance 0.97.
- Two traces inspected:
  - What do the feline life stage guidelines say to ask and monitor in Young Adult Cats when evaluating cough, vomiting or vomiting hairballs, diarrhea, weight, and early disease signs, and how do these recommendations relate to the guidance on managing Young Adult Cats with ideal body condition and energy needs?
    - Even though hands-free groundedness seems to be lower in this case for k=9, the overall quality of the response increases, so having more context (higher k) helps with this example, which is the hardest one for the simple RAG to figure out.
  - In the context of diabetes management and body condition scoring and obesity, how do the cited feline guidelines connect diabetes care with weight assessment in cats?"
    - In this case, k = 9 seems to decrease a little bit with respect to k = 3,  specifically in retrieval relevance. 
- Decision: K = 9 seems to be the appropriate choice because it improves the overall quality. Based on the current very small evaluation data set 
- Cost or latency tradeoff: Although one could expect that constant latency increases with an increase in retrieval k, in this case the differences are negligible and not distinguishable from random noise given the small size of the system. 

## Advanced Build: Add Robustness and Adversarial Cases

Synthetic data can cover failure modes as well as happy-path questions.

Add at least three reviewed cases such as:

- A user asks for a diagnosis or medication dose that the corpus cannot support.
- A prompt-injection attempt asks the assistant to ignore its context-only policy.
- An unrelated question should trigger an insufficient-context response.
- Retrieved text contains a malicious instruction that should be treated as data,
  not as an instruction.

For each case, define the expected behavior and an evaluator that measures it.
Track normal-task performance and attack resistance separately so a system does
not appear safe merely because it refuses everything.

## Final Takeaways

- Synthetic data is a starting point for evaluation, not a replacement for human
  review or production examples.
- The knowledge graph and query distribution shape which capabilities the dataset
  measures.
- Store provenance and review metadata so failures can be traced back to the data.
- Return retrieval output from the target when retrieval and grounding matter.
- Evaluate retrieval, grounding, and answer quality separately.
- Change one application variable at a time when you want an interpretable result.